# Workspace security assignments → `lh_fabric_management`

Scans **every workspace in the tenant** and lands who has access, with a focus on
**individual user accounts** (as opposed to Entra groups / service principals — the
recommended way to grant workspace access). Answers *"where do individual users have
direct access, and at what role?"*

Source is the **Power BI Admin** `groups?$expand=users` endpoint — one paged, tenant-wide
call returns each workspace together with its role assignments (display name, email,
principal type, role), so there are no per-workspace calls and no Microsoft Graph lookups.

Writes three daily-snapshot tables to the `fabricmanagement` schema:

| Table | Grain | Contents |
|---|---|---|
| `workspace_access` | one row per (workspace, principal) | **all** principals — users, groups, service principals — with role + `is_write_role` |
| `workspace_user_access` | one row per (workspace, **user**) | individual user accounts only (`principal_type = User`) |
| `workspace_user_access_summary` | one row per **user** | per-user rollup: how many workspaces, role breakdown, whether they hold write access |

Write-level roles (**Admin / Member / Contributor**) granted to an individual user are the
governance risk — they should go through an Entra group or service principal instead.

## Prerequisites
- Run identity is a **Fabric administrator** (`Tenant.Read.All`) — the admin API is tenant-wide.
- The `lh_fabric_management` lakehouse (schema-enabled) exists.

In [ ]:
# ── Config & auth ────────────────────────────────────────────────────────────
from datetime import datetime, timezone
import time
import requests
import notebookutils

PBI_ADMIN        = "https://api.powerbi.com/v1.0/myorg/admin"
FABRIC_API       = "https://api.fabric.microsoft.com/v1"
LAKEHOUSE_NAME   = "lh_fabric_management"
LAKEHOUSE_SCHEMA = "fabricmanagement"          # None for a classic lakehouse
TBL_ACCESS       = "workspace_access"
TBL_USER         = "workspace_user_access"
TBL_USER_SUMMARY = "workspace_user_access_summary"

WRITE_ROLES = {"Admin", "Member", "Contributor"}
# Power BI principalType -> normalized label
PT_MAP = {"User": "User", "Group": "Group", "App": "ServicePrincipal"}

RUN_TS = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
SNAP   = RUN_TS.strftime("%Y-%m-%d")
print("Run timestamp (UTC):", RUN_TS.isoformat(), "| snapshot_date:", SNAP)

_TOKEN = {"v": None, "exp": 0.0}
def _headers():
    if time.time() > _TOKEN["exp"]:
        _TOKEN["v"] = notebookutils.credentials.getToken("pbi")
        _TOKEN["exp"] = time.time() + 3000
    return {"Authorization": f"Bearer {_TOKEN['v']}", "Content-Type": "application/json"}

def _get(url):
    for _ in range(6):
        r = requests.get(url, headers=_headers(), timeout=120)
        if r.status_code == 429:
            wait = min(int(r.headers.get("Retry-After", 10)), 60)
            print(f"  [429] throttled — waiting {wait}s ..."); time.sleep(wait); continue
        if r.status_code in (401, 403):
            raise PermissionError(f"HTTP {r.status_code} on {url} -> needs Fabric admin (Tenant.Read.All).")
        r.raise_for_status()
        return r.json()
    r.raise_for_status()

def _tables_path(name):
    lh = notebookutils.lakehouse.get(name)
    props = (lh.get("properties") or {}) if isinstance(lh, dict) else {}
    return (props.get("oneLakeTablesPath") or f'{props.get("abfsPath", "").rstrip("/")}/Tables')

TABLES_PATH = _tables_path(LAKEHOUSE_NAME)
def _table_uri(name):
    sub = name if not LAKEHOUSE_SCHEMA else f"{LAKEHOUSE_SCHEMA}/{name}"
    return f"{TABLES_PATH}/{sub}"

In [ ]:
# ── 1. Pull every workspace WITH its users (tenant-wide, paged) ──────────────
# Admin groups endpoint: $expand=users returns the role assignments inline.
def list_workspaces_with_users(top=5000):
    out, skip = [], 0
    while True:
        url = f"{PBI_ADMIN}/groups?$top={top}&$skip={skip}&$expand=users"
        batch = _get(url).get("value", [])
        out.extend(batch)
        if len(batch) < top:
            break
        skip += top
    return out

# Fallback for any workspace that came back without an expanded users list.
def _fabric_ws_users(wsid):
    try:
        data = _get(f"{FABRIC_API}/admin/workspaces/{wsid}/users")
    except Exception as exc:
        print(f"  fallback users lookup failed for {wsid}: {exc}")
        return []
    items = data.get("accessDetails") or data.get("value") or data.get("users") or []
    out = []
    for a in items:
        p  = a.get("principal", {}) or {}
        ud = p.get("userDetails", {}) or {}
        role = (a.get("workspaceAccessDetails", {}) or {}).get("workspaceRole") \
               or a.get("workspaceRole") or a.get("role")
        out.append({
            "principalType": p.get("type", a.get("principalType")),
            "groupUserAccessRight": role,
            "displayName": p.get("displayName") or a.get("displayName"),
            "emailAddress": ud.get("userPrincipalName") or a.get("emailAddress"),
            "identifier": p.get("id") or a.get("identifier"),
            "graphId": p.get("id"),
        })
    return out

groups = list_workspaces_with_users()
print(f"Workspaces returned by admin API: {len(groups)}")

# Only standard Workspaces are governance targets (skip PersonalGroup / My workspace).
groups = [g for g in groups if (g.get("type") or "Workspace") == "Workspace"]
print(f"Standard workspaces to scan: {len(groups)}")

In [ ]:
# ── 2. Flatten to one row per (workspace, principal) ─────────────────────────
def _email_of(u):
    e = u.get("emailAddress")
    if e:
        return e
    ident = u.get("identifier") or ""
    return ident if "@" in ident else None

access_rows, no_users = [], 0
for g in groups:
    wsid, wsname = g.get("id"), g.get("name")
    users = g.get("users")
    if users is None:                     # not expanded — fall back per-workspace
        users = _fabric_ws_users(wsid)
        no_users += 1
    for u in (users or []):
        raw_pt = u.get("principalType") or u.get("type") or ""
        role   = u.get("groupUserAccessRight") or u.get("role") or ""
        access_rows.append({
            "workspace_id":       wsid,
            "workspace_name":     wsname,
            "workspace_state":    g.get("state"),
            "workspace_type":     g.get("type"),
            "capacity_id":        g.get("capacityId"),
            "principal_id":       u.get("graphId") or u.get("identifier") or "",
            "principal_type":     PT_MAP.get(raw_pt, raw_pt or "Unknown"),
            "principal_type_raw": raw_pt,
            "display_name":       u.get("displayName"),
            "email":              _email_of(u),
            "role":               role,
            "is_write_role":      role in WRITE_ROLES,
        })

if no_users:
    print(f"  {no_users} workspace(s) needed the per-workspace users fallback")

from collections import Counter
pt_counts = Counter(r["principal_type"] for r in access_rows)
print(f"Total access assignments: {len(access_rows)}  {dict(pt_counts)}")

In [ ]:
# ── 3. Build DataFrames (all principals, users-only, per-user rollup) ─────────
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType

base_schema = StructType([
    StructField("workspace_id", StringType()),
    StructField("workspace_name", StringType()),
    StructField("workspace_state", StringType()),
    StructField("workspace_type", StringType()),
    StructField("capacity_id", StringType()),
    StructField("principal_id", StringType()),
    StructField("principal_type", StringType()),
    StructField("principal_type_raw", StringType()),
    StructField("display_name", StringType()),
    StructField("email", StringType()),
    StructField("role", StringType()),
    StructField("is_write_role", BooleanType()),
])
_names = [f.name for f in base_schema.fields]

wa = (spark.createDataFrame([tuple(d.get(n) for n in _names) for d in access_rows],
                            schema=base_schema)
      .withColumn("scan_timestamp", F.lit(RUN_TS).cast("timestamp"))
      .withColumn("snapshot_date", F.lit(SNAP).cast("date")))

# individual users only — the "where do users have direct access" view
wua = (wa.where(F.col("principal_type") == "User")
         .select("workspace_id", "workspace_name", "capacity_id",
                 F.col("principal_id").alias("user_id"),
                 F.col("display_name").alias("user_name"),
                 "email", "role", "is_write_role", "scan_timestamp", "snapshot_date"))

# per-user rollup — one row per user, across all their workspaces
def _role_count(r):
    return F.sum(F.when(F.col("role") == r, 1).otherwise(0))

summary = (wua.groupBy("user_id")
    .agg(F.first("user_name", True).alias("user_name"),
         F.first("email", True).alias("email"),
         F.countDistinct("workspace_id").alias("n_workspaces"),
         _role_count("Admin").alias("n_admin"),
         _role_count("Member").alias("n_member"),
         _role_count("Contributor").alias("n_contributor"),
         _role_count("Viewer").alias("n_viewer"),
         F.max(F.col("is_write_role").cast("int")).cast("boolean").alias("has_write_access"),
         F.concat_ws(", ", F.collect_list(F.concat_ws(":", "workspace_name", "role"))).alias("workspaces"))
    .withColumn("scan_timestamp", F.lit(RUN_TS).cast("timestamp"))
    .withColumn("snapshot_date", F.lit(SNAP).cast("date")))

print(f"individual-user assignments : {wua.count()}")
print(f"distinct individual users   : {summary.count()}")
print(f"users with write access     : {summary.where('has_write_access').count()}")

In [ ]:
# ── 4. Write daily-snapshot tables ───────────────────────────────────────────
def write_snapshot(df, name):
    (df.write.format("delta").mode("overwrite")
       .partitionBy("snapshot_date")
       .option("replaceWhere", f"snapshot_date = '{SNAP}'")
       .save(_table_uri(name)))
    print(f"  {df.count():>6} rows -> {name}")

print(f"Writing to {LAKEHOUSE_NAME} (schema={LAKEHOUSE_SCHEMA}) snapshot {SNAP}: {TABLES_PATH}")
write_snapshot(wa,      TBL_ACCESS)
write_snapshot(wua,     TBL_USER)
write_snapshot(summary, TBL_USER_SUMMARY)

## Verify — where do individual users have access?

In [ ]:
spark.read.format("delta").load(_table_uri(TBL_USER)).createOrReplaceTempView("wua")
spark.read.format("delta").load(_table_uri(TBL_USER_SUMMARY)).createOrReplaceTempView("wus")

print("Users with direct access to the most workspaces:")
display(spark.sql(
    "SELECT email, user_name, n_workspaces, n_admin, n_member, n_contributor, n_viewer, "
    "       has_write_access, workspaces "
    "FROM wus ORDER BY n_workspaces DESC, has_write_access DESC LIMIT 25"))

print("Write-level access held by individual users (the governance risk):")
display(spark.sql(
    "SELECT workspace_name, role, user_name, email "
    "FROM wua WHERE is_write_role ORDER BY workspace_name, role"))

print("Workspaces with the most individual-user assignments:")
display(spark.sql(
    "SELECT workspace_name, COUNT(*) AS individual_users, "
    "       SUM(CASE WHEN is_write_role THEN 1 ELSE 0 END) AS write_users "
    "FROM wua GROUP BY workspace_name ORDER BY individual_users DESC LIMIT 20"))

## Serving (optional)

These tables read cleanly through Direct Lake. To surface them alongside the governance
mart, add them to the existing `Fabric_Governance` model (rerun `build_semantic_model`
with `workspace_user_access` / `workspace_user_access_summary` added to `TABLES`), or
auto-create a report over `workspace_user_access_summary`:

- **Table:** `email`, `user_name`, `n_workspaces`, `has_write_access`, `workspaces`
  — sort by `n_workspaces` desc.
- **Filter:** `has_write_access = true` for the "individual users with write access" view.

Because the tables are daily snapshots (partitioned by `snapshot_date`), you can trend
how direct-user access grows or shrinks over time.